In [ ]:
import numpy as np
from scipy.special import iv, kv
from scipy.optimize import root_scalar
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

def dR1dt_before(t, R1, lam, c_bar, c_B, model_G0):
    sqrt_lam = np.sqrt(lam)
    return model_G0 * c_B * sqrt_lam * iv(1, sqrt_lam * R1) / (lam * iv(0, sqrt_lam * R1)) - 0.5 * model_G0 * c_bar * R1

def condition_Rstar(R1, lam, c_bar, c_B, model_G0):
    sqrt_lam = np.sqrt(lam)
    return - model_G0 * c_B / (lam * iv(0, sqrt_lam * R1)) + model_G0 * c_B / lam - (model_G0 * c_bar * R1**2) / 4

In [ ]:
def L(lam, n_c, R0):
    sqrtlamR0 = np.sqrt(lam) * R0
    sqrtlamncR0 = np.sqrt(lam * n_c) * R0
    numerator = iv(0, sqrtlamncR0) * iv(1, sqrtlamR0) - np.sqrt(n_c) * iv(1, sqrtlamncR0) * iv(0, sqrtlamR0)
    denominator = iv(0, sqrtlamncR0) * kv(1, sqrtlamR0) + np.sqrt(n_c) * iv(1, sqrtlamncR0) * kv(0, sqrtlamR0)
    return numerator / denominator

def compute_b1(b0, lam, n_c, R0):
    return b0 * L(lam, n_c, R0)

def compute_b0(lam, n_c, c_B, R1, R0):
    sqrtlamR1 = np.sqrt(lam) * R1
    term1 = iv(0, sqrtlamR1)
    term2 = L(lam, n_c, R0) * kv(0, sqrtlamR1)
    return c_B / (term1 + term2)

def compute_A(b0, b1, lam, R0, c_bar):
    sqrtlamR0 = np.sqrt(lam) * R0
    return (R0 / np.sqrt(lam)) * (b0 * iv(1, sqrtlamR0) - b1 * kv(1, sqrtlamR0)) - (R0**2 / 2) * c_bar

def compute_B(b0, b1, lam, R0, c_bar, A):
    sqrtlamR0 = np.sqrt(lam) * R0
    return (1 / lam) * (b0 * iv(0, sqrtlamR0) + b1 * kv(0, sqrtlamR0)) - (R0**2 / 4) * c_bar - A * np.log(R0)

def final_equation(R0, lam, n_c, c_bar, c_B, R1):
    b0 = compute_b0(lam, n_c, c_B, R1, R0)
    b1 = compute_b1(b0, lam, n_c, R0)
    A = compute_A(b0, b1, lam, R0, c_bar)
    B = compute_B(b0, b1, lam, R0, c_bar, A)
    sqrtlamR1 = np.sqrt(lam) * R1
    term1 = (R1**2 / 4) * c_bar
    term2 = A * np.log(R1)
    term3 = B
    term4 = - (1 / lam) * (b0 * iv(0, sqrtlamR1) + b1 * kv(0, sqrtlamR1))
    return term1 + term2 + term3 + term4

def dR1dt_after(t, R1, lam, n_c, c_bar, c_B, model_G0):
    sol = root_scalar(final_equation, args=(lam, n_c, c_bar, c_B, R1), bracket=[1e-8, 0.99 * R1], method='brentq')
    if not sol.converged:
        raise RuntimeError(f"Failed to find R0 at t={t}, R1={R1}")
    R0 = sol.root
    b0 = compute_b0(lam, n_c, c_B, R1, R0)
    b1 = compute_b1(b0, lam, n_c, R0)
    A = compute_A(b0, b1, lam, R0, c_bar)
    sqrtlamR1 = np.sqrt(lam) * R1
    term1 = (1 / lam) * (b0 * iv(1, sqrtlamR1) - b1 * kv(1, sqrtlamR1))
    term2 = -(R1 / 2) * c_bar
    term3 = - A / R1
    return (term1 + term2 + term3) * model_G0, R0

In [ ]:
def simulate_tumor_growth(R1_0, t_end, lam, n_c, c_bar, c_B, model_G0,
                          rtol=1e-6, atol=1e-9, euler_dt=1e-4,
                          root_bracket_lower=1e-8, root_bracket_eps=0.99):
    # ---------------- stage 1 ----------------
    def ode_before(t, y):
        R1 = y[0]
        return [dR1dt_before(t, R1, lam, c_bar, c_B, model_G0)]

    def necrosis_event(t, y):
        R1 = y[0]
        return condition_Rstar(R1, lam, c_bar, c_B, model_G0)
    necrosis_event.terminal = True
    necrosis_event.direction = -1

    sol1 = solve_ivp(ode_before, (0.0, t_end), [R1_0], 
                    t_eval=np.linspace(0, t_end, int((t_end - 0.) / 0.001) + 1), max_step=1e-3,
                    method='RK45', rtol=rtol, atol=atol, events=necrosis_event, dense_output=True)

    t_vals_before = sol1.t
    R1_vals_before = sol1.y[0]
    R0_vals_before = np.zeros_like(R1_vals_before)

    if sol1.t_events[0].size == 0:
        return t_vals_before, R1_vals_before, R0_vals_before

    t_necrosis = sol1.t_events[0][0]
    R1_necrosis = float(sol1.sol(t_necrosis)[0])

    # ---------------- Make a small forward Euler step at the moment of necrosis ----------------
    dR1_at_star = c_B * (iv(1, np.sqrt(lam) * R1_necrosis) - kv(1, np.sqrt(lam) * R1_necrosis)) / (np.sqrt(lam) * iv(0, np.sqrt(lam) * R1_necrosis)) - R1_necrosis * c_bar / 2
    R1_after_euler = R1_necrosis + dR1_at_star * euler_dt
    t_after_euler = t_necrosis + euler_dt

    
    R0_init = None
    brackets = [
        (root_bracket_lower, root_bracket_eps * R1_after_euler),
        (root_bracket_lower, min(0.999 * R1_after_euler, R1_after_euler - 1e-12)),
    ]
    last_exc = None
    for a, b in brackets:
        if b <= a:
            continue
        try:
            sol = root_scalar(final_equation, args=(lam, n_c, c_bar, c_B, R1_after_euler),
                              bracket=[a, b], method='brentq', xtol=1e-12)
            if sol.converged and sol.root > 0:
                R0_init = sol.root
                break
        except Exception as e:
            last_exc = e

    if R0_init is None:
        raise RuntimeError(
            "The final_equation failed to find R0 right after necrosis occurred."
            f" Try R1_after_euler={R1_after_euler:.4e}, brackets={brackets}. "
            f" Last_exc: {last_exc}"
        )

    # ---------------- Stage 2（from t_after_euler） ----------------
    def ode_after(t, y):
        R1 = float(y[0])
        dR1, _ = dR1dt_after(t, R1, lam, n_c, c_bar, c_B, model_G0)
        return [dR1]

    sol2 = solve_ivp(ode_after, (t_after_euler, t_end), [R1_after_euler], 
                    t_eval=np.linspace(t_after_euler, t_end, int((t_end - t_after_euler) / 0.001) + 1), max_step=1e-3,
                    method='RK45', rtol=rtol, atol=atol, dense_output=True)

    t_vals_after = sol2.t
    R1_vals_after = sol2.y[0]

    R0_vals_after = np.empty_like(R1_vals_after)
    for idx, R1_val in enumerate(R1_vals_after):
        try:
            _, R0_vals_after[idx] = dR1dt_after(t_vals_after[idx], float(R1_val), lam, n_c, c_bar, c_B, model_G0)
        except Exception as e:
            raise RuntimeError(f"At stage 2, {idx}-th time, failed to solve R0: t={t_vals_after[idx]}, R1={R1_val}") from e


    t_event_block = np.array([t_necrosis, t_after_euler])
    R1_event_block = np.array([R1_necrosis, R1_after_euler])
    R0_event_block = np.array([0.0, R0_init])

    t_all = np.concatenate([t_vals_before, t_event_block, t_vals_after])
    R1_all = np.concatenate([R1_vals_before, R1_event_block, R1_vals_after])
    R0_all = np.concatenate([R0_vals_before, R0_event_block, R0_vals_after])

    return t_all, R1_all, R0_all

In [ ]:
from scipy.interpolate import splprep, splev

def compute_equiv_radius(bdry):
    if not np.allclose(bdry[0], bdry[-1]):
        bdry = np.vstack([bdry, bdry[0]])

    tck, u = splprep([bdry[:, 0], bdry[:, 1]], s=0, per=True)

    u_fine = np.linspace(0, 1, 2000)
    dx_du, dy_du = splev(u_fine, tck, der=1)

    length = np.trapezoid(np.sqrt(dx_du**2 + dy_du**2), u_fine)

    return length / (2 * np.pi)

In [ ]:
lam = 1;
c_B = 10;
c_bar = 0.5 * c_B
n_c = 1.e-3;
R1_0 = 2.2
model_G0 = 1.
t_end = 0.2

In [ ]:
exact_t, exact_R1, exact_R0 = simulate_tumor_growth(R1_0, t_end, lam, n_c, c_bar, c_B, model_G0, rtol=1e-6, atol=1e-9)

In [ ]:
import os
import numpy as np
import re 

def extract_number(filename):
    match = re.search(r'_(\d+)\.txt$', filename)
    return int(match.group(1)) if match else -1

data_dict = {
    "ctrl_points_0": [],  # bdry_points_*.txt
    "ctrl_points_1": [],  # bdry_points_*.txt
}

base_dirs = ["solutions", "points"]

for base_dir in base_dirs:
    if not os.path.exists(base_dir):
        continue
    
    for filename in sorted(os.listdir(base_dir), key=extract_number):  
        filepath = os.path.join(base_dir, filename)
        
        if filename.startswith("ctrl_points_0_"):
            data_dict["ctrl_points_0"].append(np.loadtxt(filepath))
        elif filename.startswith("ctrl_points_1_"):
            data_dict["ctrl_points_1"].append(np.loadtxt(filepath))
        
ctrl_points_0_list = [np.array(arr) for arr in data_dict["ctrl_points_0"]]
ctrl_points_1_list = [np.array(arr) for arr in data_dict["ctrl_points_1"]]

In [ ]:
radii_0 = []
radii_1 = []

for t in np.arange(len(ctrl_points_1_list)):
    bdry0 = ctrl_points_0_list[t]
    bdry1 = ctrl_points_1_list[t]

    if (len(bdry0) != 0):
        r0 = compute_equiv_radius(bdry0)
    else:
        r0 = 0
    r1 = compute_equiv_radius(bdry1)

    radii_0.append(r0)
    radii_1.append(r1)

In [ ]:
plt.figure(dpi=600)

plt.plot(exact_t, exact_R0, label='Reference R0(t)', color='red', linestyle='-', linewidth = 2, alpha = 0.5)
plt.plot(exact_t, exact_R1, label='Reference R1(t)', color='green', linestyle='-', linewidth = 2, alpha = 0.5)
t = np.loadtxt("./time_log.txt")
plt.plot(t, radii_0, label='Numerical R0(t)', color='blue', linestyle='-', linewidth = 1) 
plt.plot(t, radii_1, label='Numerical R1(t)', color='black', linestyle='-', linewidth = 1)
# plt.scatter(x_vals, radii_0, label='Numerical R0(t)', color='blue', s=5)  
# plt.scatter(x_vals, radii_1, label='Numerical R1(t)', color='black', s=5)

plt.xlabel("Time")
plt.ylabel("Radius")
plt.title("Numerical vs ODE Radii Over Time")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()